In [2]:
#import libraries
import json
import pandas as pd
import os

In [3]:
# Define your search terms
keywords = {
    "diagnosis": "glycated haemoglobin|HbA1c|A1C|oral glucose tolerance test|fasting plasma glucose",
    "management": "glycaemic control|blood pressure control|cholesterol control|self-monitoring of blood glucose|continuous glucose monitoring|foot exam|retinopathy screening|psychosocial + diabetes|mental health support + diabetes",
    "medication": (
        "Biguanides|Metformin|Sulfonylureas|Gliclazide|Glipizide|Glibenclamide|Glimepiride|"
        "Dipeptidyl Peptidase-4 Inhibitors|DPP-4 Inhibitors|Sitagliptin|Vildagliptin|Saxagliptin|"
        "Sodium-Glucose Cotransporter-2 Inhibitors|SGL-2 Inhibitors|Dapagliflozin|Empagliflozin|"
        "Glucagon-Like Peptide-1 Agonists|GLP-1 Agonists|Liraglutide|Exenatide|Dulaglutide|"
        "Semaglutide|Lixisenatide|Tirzepatide|Thiazolidinediones|Pioglitazone|Insulin"
    ),
    "diabetes": "diabetes|T1D|T2D|blood sugar|diabetic ketoacidosis|DKA|pancreas|blood glucose|diabetic|hypoglycaemia|hyperglycaemia|prediabetes|gestational diabetes",
}

exclusion_words = "insulinoma|short bowel syndrome|hypopituitarism|insipidus"

# Example usage:
print(keywords["medication"])


Biguanides|Metformin|Sulfonylureas|Gliclazide|Glipizide|Glibenclamide|Glimepiride|Dipeptidyl Peptidase-4 Inhibitors|DPP-4 Inhibitors|Sitagliptin|Vildagliptin|Saxagliptin|Sodium-Glucose Cotransporter-2 Inhibitors|SGL-2 Inhibitors|Dapagliflozin|Empagliflozin|Glucagon-Like Peptide-1 Agonists|GLP-1 Agonists|Liraglutide|Exenatide|Dulaglutide|Semaglutide|Lixisenatide|Tirzepatide|Thiazolidinediones|Pioglitazone|Insulin


In [4]:
path_training = r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\raw\benchmark\train"
path_test = r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\raw\benchmark\test"
path_evaluation = r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\raw\benchmark\evaluation"

# MedMCQA

In [11]:
#start with the train data
train_mcqa_df = pd.read_json(r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\MedMCQA\train.json", lines=True)

# Store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (train_mcqa_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (train_mcqa_df['question']).str.contains(keywords[keyword_key]) |        
        (train_mcqa_df['exp'].str.lower()).str.contains(keywords[keyword_key]) |
        (train_mcqa_df['exp']).str.contains(keywords[keyword_key]) |
        (train_mcqa_df['topic_name'].str.lower()).str.contains(keywords[keyword_key])|
        (train_mcqa_df['topic_name']).str.contains(keywords[keyword_key])
    )
    
    # Exclude rows already used
    available_mask = mask & ~train_mcqa_df.index.isin(used_indices)
    
    df_filtered = train_mcqa_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

# Combine all into one DataFrame
final_train_mcqa_df= pd.concat(filtered_dfs, ignore_index=True)

# Exclude negative texts
final_train_mcqa_df = final_train_mcqa_df[~final_train_mcqa_df['question'].str.contains(exclusion_words)]

# Set column names
print (final_train_mcqa_df.columns)
final_train_mcqa_df.columns = [
    'question', 'long_answer', 'answer', 'A', 'B', 'C', 'D',
    'subject_name', 'topic_name', 'id', 'choice_type', 'subtopic_name']

#replace numeric with alphabets
answer_options = {1:'A', 2:'B', 3:'C', 4:'D'}
final_train_mcqa_df['answer'] = final_train_mcqa_df['answer'].replace(answer_options)
final_train_mcqa_df['question_type'] = final_train_mcqa_df['choice_type'] + '-choice question'

# Group and extract desired structure
grouped_train_mcqa_df = final_train_mcqa_df.groupby(['question', 'long_answer', 'answer'], dropna=False).apply(
    lambda x: pd.Series({
        'options': x[['A', 'B', 'C', 'D']].iloc[0].to_dict(),
        'meta_info': x[['subject_name', 'topic_name', 'subtopic_name', 'question_type']].iloc[0].to_dict()
    })
).reset_index()

# save to a location
grouped_train_mcqa_df['answer'] = grouped_train_mcqa_df.apply(lambda row: row['options'][row['answer']], axis=1)
grouped_train_mcqa_df.to_json(f"{path_training}/mcqa_train.json")


# Optional: check the shape
print(train_mcqa_df.shape, grouped_train_mcqa_df .shape)

grouped_train_mcqa_df.head()

Index(['question', 'exp', 'cop', 'opa', 'opb', 'opc', 'opd', 'subject_name',
       'topic_name', 'id', 'choice_type', 'subtopic_name'],
      dtype='object')
(182822, 11) (6366, 5)


C:\Users\giftz\AppData\Local\Temp\ipykernel_18924\140228146.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_train_mcqa_df = final_train_mcqa_df.groupby(['question', 'long_answer', 'answer'], dropna=False).apply(


,question,long_answer,answer,options,meta_info
0,"""Chain of lakes"" appearance seen in:",(Chronic pancreatitis) (1127-Baily & Love 24th...,Chronic pancreatitis,"{'A': 'Acute pancreatitis', 'B': 'Chronic panc...","{'subject_name': 'Surgery', 'topic_name': 'Pan..."
1,"""Double bubble sign"" seen in","(Annular pancreas) (1120, 1198-LB) (1137-B & L...",Annular pancreas,"{'A': 'Annular pancreas', 'B': 'Ureterocele', ...","{'subject_name': 'Surgery', 'topic_name': 'Pan..."
2,"""Double bubble"" sign is feature of:",Ans. d (All of the above). (Ref. Grainger's Di...,ALL of the above,"{'A': 'Annular pancreas', 'B': 'Duodenal atres...","{'subject_name': 'Radiology', 'topic_name': 'G..."
3,"""Double bubble"" sign with absent distal bowel ...",Ans. b (Duodenal atresia) (Ref. Bailey and Lov...,Duodenal atresia,"{'A': 'Duodenal webs', 'B': 'Duodenal atresia'...","{'subject_name': 'Radiology', 'topic_name': 'G..."
4,"""Floating tooth sign"" seen in",i.e. (Histiocytosis X): (631-Basis pathology 8...,Histiocytosis-X,"{'A': 'Odontogenic myxoma', 'B': 'Ameloblastom...","{'subject_name': 'Medicine', 'topic_name': 'Mi..."


In [12]:
#start with the dev data
dev_mcqa_df = pd.read_json(r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\MedMCQA\dev.json", lines=True)

# Store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (dev_mcqa_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (dev_mcqa_df['exp'].str.lower()).str.contains(keywords[keyword_key]) |
        (dev_mcqa_df['topic_name'].str.lower()).str.contains(keywords[keyword_key])|
        (dev_mcqa_df['question']).str.contains(keywords[keyword_key]) |
        (dev_mcqa_df['exp']).str.contains(keywords[keyword_key]) |
        (dev_mcqa_df['topic_name']).str.contains(keywords[keyword_key])
    )
    
    # Exclude rows already used
    available_mask = mask & ~dev_mcqa_df.index.isin(used_indices)
    
    df_filtered = dev_mcqa_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

# Combine all into one DataFrame
final_dev_mcqa_df= pd.concat(filtered_dfs, ignore_index=True)

# Set column names
final_dev_mcqa_df.columns = [
    'question', 'long_answer', 'answer', 'A', 'B', 'C', 'D',
    'subject_name', 'topic_name', 'id', 'choice_type', 'subtopic_name']

# Exclude negative texts
final_dev_mcqa_df = final_dev_mcqa_df[~final_dev_mcqa_df['question'].str.contains(exclusion_words)]

#replace numeric with alphabets
answer_options = {1:'A', 2:'B', 3:'C', 4:'D'}
final_dev_mcqa_df['answer'] = final_dev_mcqa_df['answer'].replace(answer_options)
final_dev_mcqa_df['question_type'] = final_dev_mcqa_df['choice_type'] + '-choice question'

# Group and extract desired structure
grouped_dev_mcqa_df = final_dev_mcqa_df.groupby(['question', 'long_answer', 'answer'], dropna=False).apply(
    lambda x: pd.Series({
        'options': x[['A', 'B', 'C', 'D']].iloc[0].to_dict(),
        'meta_info': x[['subject_name', 'topic_name', 'subtopic_name', 'question_type']].iloc[0].to_dict()
    })
).reset_index()

# save to a location
grouped_dev_mcqa_df['answer'] = grouped_dev_mcqa_df.apply(lambda row: row['options'][row['answer']], axis=1)
grouped_dev_mcqa_df.to_json(f"{path_evaluation}/mcqa_dev.json")

# Optional: check the shape
print(dev_mcqa_df.shape, grouped_dev_mcqa_df.shape)

grouped_dev_mcqa_df.head()

(4183, 11) (89, 5)


C:\Users\giftz\AppData\Local\Temp\ipykernel_18924\2073695531.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_dev_mcqa_df = final_dev_mcqa_df.groupby(['question', 'long_answer', 'answer'], dropna=False).apply(


,question,long_answer,answer,options,meta_info
0,40 year old female came with fullness in upper...,NaN,Non-alcoholic steatohepatitis,"{'A': 'Biliary cirrhosis', 'B': 'Luminal throm...","{'subject_name': 'Pathology', 'topic_name': No..."
1,"A 10 year old boy is having polyuria, polydips...",Lets see each option one by one :\nartter's sy...,Diabetes insipidus,"{'A': 'Diabetes insipidus', 'B': 'Renal tubula...","{'subject_name': 'Pediatrics', 'topic_name': N..."
2,A 16 years old girl came for evaluation of pri...,Answer- B. TamoxifenMedical Treatment of PCOSE...,Tamoxifen,"{'A': 'Spironolactone', 'B': 'Tamoxifen', 'C':...","{'subject_name': 'Gynaecology & Obstetrics', '..."
3,A 32 weeks pregnant diabetic female with NST n...,Non Stress Test,Proceed to biophysical profile,"{'A': 'Induction of labour', 'B': 'CS', 'C': '...","{'subject_name': 'Gynaecology & Obstetrics', '..."
4,A 50 years okl smoker and hypeensive was diagn...,Answer-C. SIADHHyponatremia is a common electr...,SIADH,"{'A': 'Cerebral salt wasting', 'B': 'Diuretic ...","{'subject_name': 'Medicine', 'topic_name': Non..."


In [13]:
#start with the test data - No answers so exclude
test_mcqa_df = pd.read_json(r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\MedMCQA\test.json", lines=True)

# Store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (test_mcqa_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (test_mcqa_df['question']).str.contains(keywords[keyword_key]) 
    )
    
    # Exclude rows already used
    available_mask = mask & ~test_mcqa_df.index.isin(used_indices)
    
    df_filtered = test_mcqa_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

# Combine all into one DataFrame
final_test_mcqa_df= pd.concat(filtered_dfs, ignore_index=True)

# Set column names
final_test_mcqa_df.columns = [ 'question', 'A', 'B', 'C', 'D',
    'subject_name', 'topic_name', 'id', 'choice_type', 'subtopic_name']

# Exclude negative texts
final_test_mcqa_df = final_test_mcqa_df[~final_test_mcqa_df['question'].str.contains(exclusion_words)]

#replace numeric with alphabets
answer_options = {1:'A', 2:'B', 3:'C', 4:'D'}
final_test_mcqa_df['question_type'] = final_test_mcqa_df['choice_type'] + '-choice question'

# Group and extract desired structure
grouped_test_mcqa_df = final_test_mcqa_df.groupby(['question'], dropna=False).apply(
    lambda x: pd.Series({
        'options': x[['A', 'B', 'C', 'D']].iloc[0].to_dict(),
        'meta_info': x[['subject_name', 'topic_name', 'subtopic_name', 'question_type']].iloc[0].to_dict()
    })
).reset_index()

# save to a location
#grouped_test_mcqa_df['answer'] = grouped_test_mcqa_df.apply(lambda row: row['options'][row['answer']], axis=1)
grouped_test_mcqa_df.to_json(f"{path_test}/mcqa_test.json")

# Optional: check the shape
print(test_mcqa_df.shape, grouped_test_mcqa_df.shape)

grouped_test_mcqa_df.head()

(6150, 9) (54, 3)


C:\Users\giftz\AppData\Local\Temp\ipykernel_18924\2648624891.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_test_mcqa_df = final_test_mcqa_df.groupby(['question'], dropna=False).apply(


,question,options,meta_info
0,A 29-year-old woman diagnosed with type 1 diab...,"{'A': 'Worsening of diabetes', 'B': 'Lipohyper...","{'subject_name': 'Unknown', 'topic_name': nan,..."
1,A 55-year-old diabetic woman with altered sens...,"{'A': 'Autonomic neuropathy', 'B': 'Diabetic a...","{'subject_name': 'Unknown', 'topic_name': nan,..."
2,A 62-year-old diabetic female had a fall and h...,"{'A': 'Alzheimer’s disease', 'B': 'Left pariet...","{'subject_name': 'Unknown', 'topic_name': nan,..."
3,A 65 old male with a history of hypeension and...,"{'A': 'concomitant squint', 'B': 'paralytic sq...","{'subject_name': 'Ophthalmology', 'topic_name'..."
4,A 75-yrs-old woman with diabetes is talking or...,"{'A': 'Acarbose', 'B': 'Glyburide', 'C': 'Metf...","{'subject_name': 'Unknown', 'topic_name': nan,..."


In [14]:
#total extracted
final_dev_mcqa_df.shape[0] + final_train_mcqa_df.shape[0] + grouped_test_mcqa_df.shape[0]

6509

# MedQA

In [15]:
#start with the dev data
train_medqa_df = pd.read_json(r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\MedQA\data_clean\questions\US\train.jsonl", lines=True)

#store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (train_medqa_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (train_medqa_df['question']).str.contains(keywords[keyword_key]) |
        (train_medqa_df['options']).str.contains(keywords[keyword_key])
    )
    
    # Exclude rows already used
    available_mask = mask & ~train_medqa_df.index.isin(used_indices)
    
    df_filtered = train_medqa_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

#combine all into one DataFrame
final_train_medqa_df= pd.concat(filtered_dfs, ignore_index=True)
final_train_medqa_df['id'] = final_train_medqa_df['subtopic_name'].str.upper() + final_train_medqa_df.index.astype(str) +'MEDQA'

# Exclude negative texts
final_train_medqa_df = final_train_medqa_df[~final_train_medqa_df['question'].str.contains(exclusion_words)]

#change column names
final_train_medqa_df.rename(columns={'meta_info':'process_step'}, inplace=True)
final_train_medqa_df['question_type'] = "multiple-choice question"


#group and extract desired structure
grouped_train_medqa_df = final_train_medqa_df.groupby(
    ['question', 'answer'], dropna=False
).apply(
    lambda x: pd.Series({
        'options': x['options'].iloc[0],  # assumes options are consistent per group
        'meta_info': x[['process_step', 'subtopic_name', 'question_type']].iloc[0].to_dict()
    })
).reset_index()

# save to a location
grouped_train_medqa_df.to_json(f"{path_training}/medqa_train.json")

#optional: check the shape
print(train_medqa_df.shape, grouped_train_medqa_df.shape)

grouped_train_medqa_df.head()


(10178, 5) (1145, 4)


C:\Users\giftz\AppData\Local\Temp\ipykernel_18924\497135999.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(


,question,answer,options,meta_info
0,"A 50-year-old man presents with headache, che...",Smooth muscle hyperplasia and duplication of t...,{'A': 'Smooth muscle hyperplasia and duplicati...,"{'process_step': 'step1', 'subtopic_name': 'di..."
1,A 12-hour-old newborn is found to have difficu...,Prostaglandins E2,"{'A': 'Indomethacin', 'B': 'Low-dose aspirin',...","{'process_step': 'step1', 'subtopic_name': 'di..."
2,A 12-hour-old newborn is urgently transferred ...,Vascular vasodilation,"{'A': 'Decreased platelet aggregation', 'B': '...","{'process_step': 'step1', 'subtopic_name': 'di..."
3,A 12-year-old boy presents with progressive cl...,Friedreich’s ataxia,"{'A': 'Ataxia-telangiectasia', 'B': 'Charcot-M...","{'process_step': 'step2&3', 'subtopic_name': '..."
4,A 12-year-old girl comes to the clinic with a ...,Von-Gierke's disease,"{'A': 'Hereditary hemochromatosis', 'B': 'Cori...","{'process_step': 'step1', 'subtopic_name': 'di..."


In [16]:
#start with the dev data
dev_medqa_df = pd.read_json(r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\MedQA\data_clean\questions\US\dev.jsonl", lines=True)

#store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (dev_medqa_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (dev_medqa_df['question']).str.contains(keywords[keyword_key]) |
        (dev_medqa_df['options']).str.contains(keywords[keyword_key])
    )
    
    # Exclude rows already used
    available_mask = mask & ~dev_medqa_df.index.isin(used_indices)
    
    df_filtered = dev_medqa_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

#combine all into one DataFrame
final_dev_medqa_df= pd.concat(filtered_dfs, ignore_index=True)

# Exclude negative texts
final_dev_medqa_df = final_dev_medqa_df[~final_dev_medqa_df['question'].str.contains(exclusion_words)]

#change column names
final_dev_medqa_df.rename(columns={'meta_info':'process_step'}, inplace=True)
final_dev_medqa_df['question_type'] = "multiple-choice question"


#group and extract desired structure
grouped_dev_medqa_df = final_dev_medqa_df.groupby(
    ['question', 'answer'], dropna=False
).apply(
    lambda x: pd.Series({
        'options': x['options'].iloc[0],  # assumes options are consistent per group
        'meta_info': x[['process_step', 'subtopic_name', 'question_type']].iloc[0].to_dict()
    })
).reset_index()

# save to a location
grouped_dev_medqa_df.to_json(f"{path_evaluation}/medqa_dev.json")


#optional: check the shape
print(dev_medqa_df.shape, grouped_dev_medqa_df.shape)


grouped_dev_medqa_df.head()


(1272, 5) (117, 4)


C:\Users\giftz\AppData\Local\Temp\ipykernel_18924\3714514088.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(


,question,answer,options,meta_info
0,A 14-year-old boy presents to his primary care...,Metoprolol,"{'A': 'Metformin', 'B': 'Metoprolol', 'C': 'Hy...","{'process_step': 'step2&3', 'subtopic_name': '..."
1,A 14-year-old girl with a BMI of 17 complains ...,Enhances potassium uptake,{'A': 'Down-regulates glycogen synthase activi...,"{'process_step': 'step1', 'subtopic_name': 'di..."
2,A 19-year-old woman comes to the physician for...,Insulin glargine,"{'A': 'Insulin glargine', 'B': 'Insulin glulis...","{'process_step': 'step1', 'subtopic_name': 'di..."
3,A 21-year-old female college student comes to ...,Diabetes mellitus,"{'A': 'Smoking history', 'B': 'Frequent vagina...","{'process_step': 'step2&3', 'subtopic_name': '..."
4,A 22-year-old female software analyst presents...,Hypothyroidism,"{'A': 'Hypothyroidism', 'B': 'Acute intermitte...","{'process_step': 'step1', 'subtopic_name': 'di..."


In [17]:
#start with the test data
test_medqa_df = pd.read_json(r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\MedQA\data_clean\questions\US\test.jsonl", lines=True)

#store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (test_medqa_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (test_medqa_df['question']).str.contains(keywords[keyword_key]) |
        (test_medqa_df['options']).str.contains(keywords[keyword_key])
    )
    
    # Exclude rows already used
    available_mask = mask & ~test_medqa_df.index.isin(used_indices)
    
    df_filtered = test_medqa_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

#combine all into one DataFrame
final_test_medqa_df= pd.concat(filtered_dfs, ignore_index=True)

# Exclude negative texts
final_test_medqa_df = final_test_medqa_df[~final_test_medqa_df['question'].str.contains(exclusion_words)]

#change column names
final_test_medqa_df.rename(columns={'meta_info':'process_step'}, inplace=True)
final_test_medqa_df['question_type'] = "multiple-choice question"


#group and extract desired structure
grouped_test_medqa_df = final_test_medqa_df.groupby(
    ['question', 'answer'], dropna=False
).apply(
    lambda x: pd.Series({
        'options': x['options'].iloc[0],  # assumes options are consistent per group
        'meta_info': x[['process_step', 'subtopic_name', 'question_type']].iloc[0].to_dict()
    })
).reset_index()

# save to a location
grouped_test_medqa_df.to_json(f"{path_test}/medqa_test.json")


#optional: check the shape
print(test_medqa_df.shape, grouped_test_medqa_df.shape)


grouped_test_medqa_df.head()


(1273, 5) (133, 4)


C:\Users\giftz\AppData\Local\Temp\ipykernel_18924\56911822.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(


,question,answer,options,meta_info
0,A 13-year-old boy is brought to his pediatrici...,Glandular tissue enlargement,"{'A': 'Klinefelter syndrome', 'B': 'Glandular ...","{'process_step': 'step2&3', 'subtopic_name': '..."
1,A 13-year-old boy is brought to the emergency ...,Decreased glucose uptake by adipocytes,{'A': 'Decreased glucose uptake by adipocytes'...,"{'process_step': 'step1', 'subtopic_name': 'di..."
2,A 14-year-old boy presents to the emergency de...,Radiograph of the wrist,"{'A': 'Reassurance and pain management', 'B': ...","{'process_step': 'step2&3', 'subtopic_name': '..."
3,A 15-year-old girl comes to the physician beca...,Anovulation,"{'A': 'Uterine fibroids', 'B': 'Congenital def...","{'process_step': 'step2&3', 'subtopic_name': '..."
4,A 16-year-old boy is rushed to the emergency r...,Administer IV fluids and insulin,"{'A': 'Administer IV fluids and insulin', 'B':...","{'process_step': 'step2&3', 'subtopic_name': '..."


# MMLU 

In [18]:
# Path to your folder
folder_path = r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\MMLU\data\auxiliary_train"

# List all CSV files in the folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

# Read all CSVs into a list of DataFrames
dfs = [pd.read_csv(os.path.join(folder_path, file), header=None) for file in csv_files]

# Optional: Concatenate all into one DataFrame
train_mmlu_df = pd.concat(dfs, ignore_index=True)
train_mmlu_df.columns = ['question', 'A', 'B', 'C', 'D', 'answer']
train_mmlu_df['question_type'] = "multiple-choice question"

#store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (train_mmlu_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (train_mmlu_df['question']).str.contains(keywords[keyword_key]) |
        (train_mmlu_df['A'].str.lower()).str.contains(keywords[keyword_key]) |
        (train_mmlu_df['A']).str.contains(keywords[keyword_key]) |
        (train_mmlu_df['B'].str.lower()).str.contains(keywords[keyword_key]) |
        (train_mmlu_df['B']).str.contains(keywords[keyword_key]) |
        (train_mmlu_df['C'].str.lower()).str.contains(keywords[keyword_key]) |
        (train_mmlu_df['C']).str.contains(keywords[keyword_key]) |
        (train_mmlu_df['D'].str.lower()).str.contains(keywords[keyword_key]) |
        (train_mmlu_df['D']).str.contains(keywords[keyword_key]) 
    )
    
    # Exclude rows already used
    available_mask = mask & ~train_mmlu_df.index.isin(used_indices)
    
    df_filtered = train_mmlu_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)


#combine all into one DataFrame with ideal format
final_train_mmlu_df= pd.concat(filtered_dfs, ignore_index=True).drop_duplicates()

# Exclude negative texts
final_train_mmlu_df = final_train_mmlu_df[~final_train_mmlu_df['question'].str.contains(exclusion_words)]

grouped_train_mmlu_df = final_train_mmlu_df.groupby(['question', 'answer'], dropna=False).apply(
    lambda x: pd.Series({
    'options': x[['A','B', 'C', 'D']].iloc[0].to_dict(),
    'meta_info': x[['subtopic_name', 'question_type']].iloc[0].to_dict()
    })
).reset_index()

# save to a location
grouped_train_mmlu_df['answer'] = grouped_train_mmlu_df.apply(lambda row: row['options'][row['answer']], axis=1)
grouped_train_mmlu_df.to_json(f"{path_training}/mmlu_train.json")

print(final_train_mmlu_df.shape, train_mmlu_df.shape )
grouped_train_mmlu_df.head()

(424, 8) (99842, 7)


C:\Users\giftz\AppData\Local\Temp\ipykernel_18924\1055423431.py:50: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_train_mmlu_df = final_train_mmlu_df.groupby(['question', 'answer'], dropna=False).apply(


,question,answer,options,meta_info
0,"""Your son is one of the sickest kids we've eve...",desperate and fearful,"{'A': 'desperate and fearful', 'B': 'special a...","{'subtopic_name': 'diabetes', 'question_type':..."
1,"""Your son is one of the sickest kids we've eve...",treating a sick son needs great efforts physic...,{'A': 'long distances of marathon made him wor...,"{'subtopic_name': 'diabetes', 'question_type':..."
2,(Reuters)--- A Michigan man credited his dog w...,saved his master's life,"{'A': 'is in top physical condition', 'B': 'wa...","{'subtopic_name': 'diabetes', 'question_type':..."
3,(Reuters)--- A Michigan man credited his dog w...,Douthett's wife felt something about his disease,"{'A': 'Douthett's wife was a doctor', 'B': 'Do...","{'subtopic_name': 'diabetes', 'question_type':..."
4,(Reuters)--- A Michigan man credited his dog w...,The couple shouldn't have to pay the co-pay be...,{'A': 'It is hard for the couple to explain th...,"{'subtopic_name': 'diabetes', 'question_type':..."


In [19]:
# Path to your folder
folder_path = r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\MMLU\data\dev"

# List all CSV files in the folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

# Read all CSVs into a list of DataFrames
dfs = [pd.read_csv(os.path.join(folder_path, file), header=None) for file in csv_files]

# Optional: Concatenate all into one DataFrame
dev_mmlu_df = pd.concat(dfs, ignore_index=True)
dev_mmlu_df.columns = ['question', 'A', 'B', 'C', 'D', 'answer']

#store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (dev_mmlu_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (dev_mmlu_df['question']).str.contains(keywords[keyword_key]) |
        (dev_mmlu_df['A'].str.lower()).str.contains(keywords[keyword_key]) |
        (dev_mmlu_df['A']).str.contains(keywords[keyword_key]) |
        (dev_mmlu_df['B'].str.lower()).str.contains(keywords[keyword_key]) |
        (dev_mmlu_df['B']).str.contains(keywords[keyword_key]) |
        (dev_mmlu_df['C'].str.lower()).str.contains(keywords[keyword_key]) |
        (dev_mmlu_df['C']).str.contains(keywords[keyword_key]) |
        (dev_mmlu_df['D'].str.lower()).str.contains(keywords[keyword_key]) |
        (dev_mmlu_df['D']).str.contains(keywords[keyword_key]) 
    )
    
    # Exclude rows already used
    available_mask = mask & ~dev_mmlu_df.index.isin(used_indices)
    
    df_filtered = dev_mmlu_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

#combine all into one DataFrame
final_dev_mmlu_df = pd.concat(filtered_dfs, ignore_index=True).drop_duplicates()
final_dev_mmlu_df['question_type'] = 'multiple-choice question'

# Exclude negative texts
final_dev_mmlu_df = final_dev_mmlu_df[~final_dev_mmlu_df['question'].str.contains(exclusion_words)]

grouped_dev_mmlu_df = final_dev_mmlu_df.groupby(['question', 'answer'], dropna=False).apply(
    lambda x: pd.Series({
    'options': x[['A','B', 'C', 'D']].iloc[0].to_dict(),
    'meta_info': x[['subtopic_name', 'question_type']].iloc[0].to_dict()
    })
).reset_index()

grouped_dev_mmlu_df['answer'] = grouped_dev_mmlu_df.apply(lambda row: row['options'][row['answer']], axis=1)
grouped_dev_mmlu_df.to_json(f"{path_evaluation}/mmlu_dev.json")

print(final_dev_mmlu_df.shape, dev_mmlu_df.shape )
grouped_dev_mmlu_df.head()

(3, 8) (285, 6)


C:\Users\giftz\AppData\Local\Temp\ipykernel_18924\2343271495.py:49: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_dev_mmlu_df = final_dev_mmlu_df.groupby(['question', 'answer'], dropna=False).apply(


,question,answer,options,meta_info
0,"In a cohort study, the risk ratio of developin...",Based on the information given we cannot tell ...,{'A': 'The tea drinkers have lower risk of dev...,"{'subtopic_name': 'diabetes', 'question_type':..."
1,What is the first-line drug for patients with ...,Metformin,"{'A': 'Acarbose', 'B': 'Metformin', 'C': 'Sulp...","{'subtopic_name': 'medication', 'question_type..."
2,Which of the following is not a true statement?,Insulin promotes glucose uptake by all tissues...,{'A': 'Muscle glycogen is broken down enzymati...,"{'subtopic_name': 'medication', 'question_type..."


In [20]:
# Path to your folder
folder_path = r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\MMLU\data\val"

# List all CSV files in the folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

# Read all CSVs into a list of DataFrames
dfs = [pd.read_csv(os.path.join(folder_path, file), header=None) for file in csv_files]

# Optional: Concatenate all into one DataFrame
val_mmlu_df = pd.concat(dfs, ignore_index=True)
val_mmlu_df.columns = ['question', 'A', 'B', 'C', 'D', 'answer']

#store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (val_mmlu_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (val_mmlu_df['question']).str.contains(keywords[keyword_key]) |
        (val_mmlu_df['A'].str.lower()).str.contains(keywords[keyword_key]) |
        (val_mmlu_df['A']).str.contains(keywords[keyword_key]) |
        (val_mmlu_df['B'].str.lower()).str.contains(keywords[keyword_key]) |
        (val_mmlu_df['B']).str.contains(keywords[keyword_key]) |
        (val_mmlu_df['C'].str.lower()).str.contains(keywords[keyword_key]) |
        (val_mmlu_df['C']).str.contains(keywords[keyword_key]) |
        (val_mmlu_df['D'].str.lower()).str.contains(keywords[keyword_key]) |
        (val_mmlu_df['D']).str.contains(keywords[keyword_key]) 
    )
    
    # Exclude rows already used
    available_mask = mask & ~val_mmlu_df.index.isin(used_indices)
    
    df_filtered = val_mmlu_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

#combine all into one DataFrame
final_val_mmlu_df= pd.concat(filtered_dfs, ignore_index=True).drop_duplicates()
final_val_mmlu_df['question_type'] = 'multiple-choice question'

# Exclude negative texts
final_val_mmlu_df = final_val_mmlu_df[~final_val_mmlu_df['question'].str.contains(exclusion_words)]

grouped_val_mmlu_df = final_val_mmlu_df.groupby(['question', 'answer'], dropna=False).apply(
    lambda x: pd.Series({
    'options': x[['A','B', 'C', 'D']].iloc[0].to_dict(),
    'meta_info': x[['subtopic_name', 'question_type']].iloc[0].to_dict()
    })
).reset_index()
grouped_val_mmlu_df['answer'] = grouped_val_mmlu_df.apply(lambda row: row['options'][row['answer']], axis=1)
grouped_val_mmlu_df.to_json(f"{path_evaluation}/mmlu_val.json")

print(final_val_mmlu_df.shape, val_mmlu_df.shape )
grouped_val_mmlu_df.head()

(12, 8) (1531, 6)


C:\Users\giftz\AppData\Local\Temp\ipykernel_18924\2206678904.py:49: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_val_mmlu_df = final_val_mmlu_df.groupby(['question', 'answer'], dropna=False).apply(


,question,answer,options,meta_info
0,A 14-year-old girl with a 9-year history of ty...,Countertransference,"{'A': 'Countertransference', 'B': 'Identificat...","{'subtopic_name': 'diabetes', 'question_type':..."
1,A 52-year-old woman begins pharmacotherapy aft...,Metformin,"{'A': 'Acarbose', 'B': 'Glyburide', 'C': 'Metf...","{'subtopic_name': 'medication', 'question_type..."
2,A 63-year-old man is brought to the emergency ...,Hypercoagulability from advanced malignancy,"{'A': 'Carcinoid syndrome', 'B': 'Hypercoagula...","{'subtopic_name': 'diabetes', 'question_type':..."
3,A person suffering from severe dehydration and...,Insulin,"{'A': 'ADH', 'B': 'Cortisol', 'C': 'Aldosteron...","{'subtopic_name': 'medication', 'question_type..."
4,An observational study in diabetics assesses t...,(40*70)/(20*60),"{'A': '(40/30)/(20/40)', 'B': '(40*40)/(20*30)...","{'subtopic_name': 'diabetes', 'question_type':..."


In [23]:
# Path to your folder
folder_path = r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\MMLU\data\test"

# List all CSV files in the folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

# Read all CSVs into a list of DataFrames
dfs = [pd.read_csv(os.path.join(folder_path, file), header=None) for file in csv_files]

# Optional: Concatenate all into one DataFrame
test_mmlu_df = pd.concat(dfs, ignore_index=True)
test_mmlu_df.columns = ['question', 'A', 'B', 'C', 'D', 'answer']

#store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (test_mmlu_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (test_mmlu_df['question']).str.contains(keywords[keyword_key]) |
        (test_mmlu_df['A'].str.lower()).str.contains(keywords[keyword_key]) |
        (test_mmlu_df['A']).str.contains(keywords[keyword_key]) |
        (test_mmlu_df['B'].str.lower()).str.contains(keywords[keyword_key]) |
        (test_mmlu_df['B']).str.contains(keywords[keyword_key]) |
        (test_mmlu_df['C'].str.lower()).str.contains(keywords[keyword_key]) |
        (test_mmlu_df['C']).str.contains(keywords[keyword_key]) |
        (test_mmlu_df['D'].str.lower()).str.contains(keywords[keyword_key]) |
        (test_mmlu_df['D']).str.contains(keywords[keyword_key]) 
    )
    
    # Exclude rows already used
    available_mask = mask & ~test_mmlu_df.index.isin(used_indices)
    
    df_filtered = test_mmlu_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

#combine all into one DataFrame
final_test_mmlu_df= pd.concat(filtered_dfs, ignore_index=True).drop_duplicates()
final_test_mmlu_df['question_type'] = 'multiple-choice question'

# Exclude negative texts
final_test_mmlu_df = final_test_mmlu_df[~final_test_mmlu_df['question'].str.contains(exclusion_words)]

grouped_test_mmlu_df = final_test_mmlu_df.groupby(['question', 'answer'], dropna=False).apply(
    lambda x: pd.Series({
    'options': x[['A','B', 'C', 'D']].iloc[0].to_dict(),
    'meta_info': x[['subtopic_name', 'question_type']].iloc[0].to_dict()
    })
).reset_index()
grouped_test_mmlu_df['answer'] =grouped_test_mmlu_df.apply(lambda row: row['options'][row['answer']], axis=1)
grouped_test_mmlu_df.to_json(f"{path_test}/mmlu_test.json")

print(final_test_mmlu_df.shape, test_mmlu_df.shape )
grouped_test_mmlu_df.head()


(62, 8) (14042, 6)


C:\Users\giftz\AppData\Local\Temp\ipykernel_18924\240421213.py:49: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_test_mmlu_df = final_test_mmlu_df.groupby(['question', 'answer'], dropna=False).apply(


,question,answer,options,meta_info
0,A 12-year-old boy with sickle cell disease and...,His sickle cell disease is affecting his hemog...,{'A': 'His daily glucose control is better tha...,"{'subtopic_name': 'diabetes', 'question_type':..."
1,A 22-year-old woman comes to the physician in ...,Influenza virus vaccine,"{'A': 'Dietary modification for weight loss', ...","{'subtopic_name': 'diabetes', 'question_type':..."
2,"A 29-year-old Hispanic woman, gravida 3, para ...",Recommend beginning a weight loss program,{'A': 'Administer an epidural injection of met...,"{'subtopic_name': 'diabetes', 'question_type':..."
3,A 31-year-old woman with type 2 diabetes melli...,α-Toxin,"{'A': 'Fimbriae', 'B': 'Pneumolysin', 'C': 'Po...","{'subtopic_name': 'diabetes', 'question_type':..."
4,A 37-year-old man with type 1 diabetes mellitu...,Adhere to the patient's wishes and discuss hom...,{'A': 'Adhere to the patient's wishes and disc...,"{'subtopic_name': 'diabetes', 'question_type':..."


In [24]:
final_train_mmlu_df.shape[0] + final_test_mmlu_df.shape[0] + final_val_mmlu_df.shape[0] +  final_dev_mmlu_df.shape[0]

501

# PubMedQA

In [25]:
#start with the dev data
pubmedqa_pqau_df = pd.read_json(r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\PubMedQA\PubMedQA ori_pqau.json")

pubmedqa_pqau_df = pubmedqa_pqau_df.T.reset_index()
pubmedqa_pqau_df.columns = ['id', 'question', 'context', 'labels', 'meshes', 'year', 'long_answer']

#store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (pubmedqa_pqau_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (pubmedqa_pqau_df['question']).str.contains(keywords[keyword_key]) |
        (pubmedqa_pqau_df['context']).str.contains(keywords[keyword_key]) |
        (pubmedqa_pqau_df['long_answer'].str.lower()).str.contains(keywords[keyword_key]) |
        (pubmedqa_pqau_df['long_answer']).str.contains(keywords[keyword_key]) |
        (pubmedqa_pqau_df['meshes']).str.contains(keywords[keyword_key])
    )
    
    # Exclude rows already used
    available_mask = mask & ~pubmedqa_pqau_df.index.isin(used_indices)
    
    df_filtered = pubmedqa_pqau_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

#combine all into one DataFrame
final_pubmedqa_pqau_df = pd.concat(filtered_dfs, ignore_index=True)
final_pubmedqa_pqau_df['question_type'] = 'long answer question'

# Exclude negative texts
final_pubmedqa_pqau_df = final_pubmedqa_pqau_df[~final_pubmedqa_pqau_df['question'].str.contains(exclusion_words)]

#extract desired structure?????
final_pubmedqa_pqau_df.to_json(f"{path_training}/unanswered/pubmedpqau.json")

#optional: check the shape
print(final_pubmedqa_pqau_df.shape, pubmedqa_pqau_df.shape)
final_pubmedqa_pqau_df.head()

OSError: Cannot save file into a non-existent directory: 'C:\Users\giftz\OneDrive\MIT\MIT 807\data\raw\benchmark\train\unanswered'

In [26]:
#start with the dev data
pubmedqa_pqaa_df = pd.read_json(r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\PubMedQA\PubMedQA ori_pqaa.json")

pubmedqa_pqaa_df = pubmedqa_pqaa_df.T.reset_index()
pubmedqa_pqaa_df.columns = ['id', 'question', 'context', 'labels', 'long_answer', 'meshes',
       'final_decision']

#store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (pubmedqa_pqaa_df['Squestion'].str.lower()).str.contains(keywords[keyword_key]) |
        (pubmedqa_pqaa_df['question']).str.contains(keywords[keyword_key]) |
        pubmedqa_pqaa_df['context'].str.contains(keywords[keyword_key]) |
        (pubmedqa_pqaa_df['long_answer'].str.lower()).str.contains(keywords[keyword_key]) |
        (pubmedqa_pqaa_df['long_answer']).str.contains(keywords[keyword_key]) |
        pubmedqa_pqaa_df['meshes'].str.contains(keywords[keyword_key])
    )
    
    # Exclude rows already used
    available_mask = mask & ~pubmedqa_pqaa_df.index.isin(used_indices)
    
    df_filtered = pubmedqa_pqaa_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

#combine all into one DataFrame
final_pubmedqa_pqaa_df= pd.concat(filtered_dfs, ignore_index=True)
final_pubmedqa_pqaa_df['question_type'] = "long answer question"
final_pubmedqa_pqaa_df['options'] = [{'A':'yes','B':'no','C':'maybe'}] * len(final_pubmedqa_pqaa_df)

# Exclude negative texts
final_pubmedqa_pqaa_df = final_pubmedqa_pqaa_df[~final_pubmedqa_pqaa_df['question'].str.contains(exclusion_words)]
final_pubmedqa_pqaa_df = final_pubmedqa_pqaa_df.rename(columns={'final_decision':'answer'})
#extract desired structure?????
final_pubmedqa_pqaa_df.to_json(f"{path_training}/pubmedpqaa.json")

#optional: check the shape|
print(final_pubmedqa_pqaa_df.shape)S
final_pubmedqa_pqaa_df.head()

(9361, 10)


,id,question,context,labels,long_answer,meshes,answer,subtopic_name,question_type,options
0,25424745,Do adolescents and young adults with type 1 di...,[Little is known about endothelial function in...,"[OBJECTIVE, METHODS, RESULTS]",Adolescents with type 1 diabetes displayed evi...,"[Adolescent, Carotid Arteries, Diabetes Mellit...",yes,diagnosis,long answer question,"{'A': 'yes', 'B': 'no', 'C': 'maybe'}"
1,25440345,Is higher fasting glucose next day after intra...,[We aimed to test the outcome-predictive power...,"[BACKGROUND, METHODS, RESULTS]",FG is a powerful predictor associated with the...,"[Aged, Blood Glucose, Brain Ischemia, Fasting,...",yes,diagnosis,long answer question,"{'A': 'yes', 'B': 'no', 'C': 'maybe'}"
2,27502250,Is glycemic variability evaluated by continuou...,[The present study aimed to identify the relat...,"[BACKGROUND, METHODS, RESULTS]",GV predicts independently the 10-y CVD risk of...,"[Adult, Blood Glucose, Blood Glucose Self-Moni...",yes,diagnosis,long answer question,"{'A': 'yes', 'B': 'no', 'C': 'maybe'}"
3,25071077,Is insulin sensitivity an important determinan...,[Diabetic nephropathy (DN) remains the most co...,"[OBJECTIVE, METHODS, RESULTS]",A significant proportion of adolescents with t...,"[Adolescent, Albuminuria, Biomarkers, Child, C...",yes,diagnosis,long answer question,"{'A': 'yes', 'B': 'no', 'C': 'maybe'}"
4,24773597,Does frequency of mealtime insulin bolus predi...,"[Within pediatric diabetes management, two ele...","[BACKGROUND, METHODS, RESULTS]","In a prospective, longitudinal design, youths'...","[Adolescent, Adolescent Behavior, Blood Glucos...",yes,diagnosis,long answer question,"{'A': 'yes', 'B': 'no', 'C': 'maybe'}"


In [27]:
final_pubmedqa_pqaa_df.shape[0] + final_pubmedqa_pqau_df.shape[0]

11049

# SQuAD

In [28]:
squad_dev_df = pd.read_json(r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\SQuAD\dev-v2.0.json")
squad_dev_df = pd.json_normalize(squad_dev_df['data'],
                  record_path=['paragraphs', 'qas'],
                  meta=['title'])

#store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (squad_dev_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (squad_dev_df['question']).str.contains(keywords[keyword_key]) |
        squad_dev_df['answers'].str.contains(keywords[keyword_key]) |
        squad_dev_df['plausible_answers'].str.contains(keywords[keyword_key])
    )
    
    # Exclude rows already used
    available_mask = mask & ~squad_dev_df.index.isin(used_indices)
    
    df_filtered = squad_dev_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

#combine all into one DataFrame
final_squad_dev_df = pd.concat(filtered_dfs, ignore_index=True)

#save
final_squad_dev_df.to_json(f"{path_evaluation}/squad_dev.json")

print(final_squad_dev_df.shape, squad_dev_df.shape)
final_squad_dev_df.head()

(0, 7) (11873, 6)


,question,id,answers,is_impossible,plausible_answers,title,subtopic_name


In [29]:
squad_train_df = pd.read_json(r"C:\Users\giftz\OneDrive\MIT\MIT 807\data\Benchmark\SQuAD\train-v2.0.json")
squad_train_df = pd.json_normalize(squad_train_df['data'],
                  record_path=['paragraphs', 'qas'],
                  meta=['title'])

#store filtered DataFrames in a list
used_indices = set()
filtered_dfs = []

for keyword_key in keywords.keys():
    mask = (
        (squad_train_df['question'].str.lower()).str.contains(keywords[keyword_key]) |
        (squad_train_df['question']).str.contains(keywords[keyword_key]) |
        squad_train_df['answers'].str.contains(keywords[keyword_key]) |
        squad_train_df['plausible_answers'].str.contains(keywords[keyword_key])
    )
    
    # Exclude rows already used
    available_mask = mask & ~squad_train_df.index.isin(used_indices)
    
    df_filtered = squad_train_df[available_mask].copy()
    df_filtered['subtopic_name'] = keyword_key
    
    # Update used indices
    used_indices.update(df_filtered.index)
    filtered_dfs.append(df_filtered)

#combine all into one DataFrame
final_squad_train_df = pd.concat(filtered_dfs, ignore_index=True)

#save
final_squad_train_df.to_json(f"{path_training}/squad_train.json")

print(final_squad_train_df.shape, squad_train_df.shape)
final_squad_train_df.head()

(20, 7) (130319, 6)


,question,id,answers,is_impossible,plausible_answers,title,subtopic_name
0,Which Iranian scientist discovered HbA1c to fu...,573033c3a23a5019007fcf74,"[{'text': 'Samuel Rahbar', 'answer_start': 393}]",False,NaN,Iran,diagnosis
1,Insulin resistance has been strongly linked to...,57271739f1498d1400e8f386,"[{'text': 'obesity', 'answer_start': 13}]",False,NaN,Nutrition,medication
2,Which contestant had Type-I diabetes?,56db2664e7c41114004b4e17,"[{'text': 'Crystal Bowersox', 'answer_start': 0}]",False,NaN,American_Idol,diabetes
3,Deletion of what gene in himans causes diabetes?,5a221a05819328001af389fd,[],True,"[{'text': 'Rev-ErbA alpha clock gene', 'answer...",Circadian_rhythm,diabetes
4,Whar does research show happens to blood gluco...,5a8cec6ffd22b3001a8d9026,[],True,[{'text': 'levels that have increased drastica...,Memory,diabetes
